In [ ]:
# ============================================================
#  RICE PEST DETECTION - FULL BACKEND (YOLOv11) - 1 cell Colab
#  Runtime -> Change runtime type -> T4 GPU, roi bam Run
# ============================================================

# ---------------- CAU HINH (sua o day) ----------------
MODEL_DIR = "/content/drive/MyDrive/rice_pest_detection/runs/yolo11s/weights"   # folder chua best.pt (YOLO11s)
DATA_DIR  = "/content/drive/MyDrive/rice_pest_detection/backend_data"   # noi luu DB + anh ket qua (tren Drive, khong mat)
API_KEY   = ""       # de trong = khong khoa. Dat vd "abc123" thi moi request phai kem header x-api-key (hoac ?key=abc123)
CONF, IMGSZ, PORT, HEARTBEAT = 0.4, 640, 8000, 10

# Thong tin phong tri: key = TEN LOP dung nhu trong model (xem log buoc 3). Tu dien them cho dung de tai cua ban.
PEST_INFO = {
    # "ray_nau": {"name": "Rầy nâu", "description": "...", "prevention": "..."},
}
# -------------------------------------------------------

import os, re, io, csv, sys, json, time, glob, uuid, base64, sqlite3, threading, subprocess, collections
from datetime import datetime
T0 = time.time()

def log(msg, tag="INFO"):
    print(f"[{time.strftime('%H:%M:%S')}] [{time.time()-T0:7.1f}s] [{tag}] {msg}", flush=True)

def run(cmd, desc):
    log(desc, "SETUP")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0: log(f"Loi: {r.stderr[-500:]}", "ERROR")

# ---------------- 1. CAI DAT ----------------
log("=== BUOC 1/7: Cai thu vien ===", "STEP")
run("pip -q install -U ultralytics fastapi 'uvicorn[standard]' python-multipart requests", "pip install ...")
if not os.path.exists("/content/cloudflared"):
    run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared", "Tai cloudflared")
log("Cai dat xong", "OK")

# ---------------- 2. DRIVE ----------------
log("=== BUOC 2/7: Mount Google Drive ===", "STEP")
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DATA_DIR}/images", exist_ok=True)
os.makedirs("/content/jobs", exist_ok=True)
log(f"Du lieu se luu o: {DATA_DIR}", "OK")

# ---------------- 3. MODEL ----------------
log("=== BUOC 3/7: Load model YOLOv11 ===", "STEP")
cands = glob.glob(f"{MODEL_DIR}/**/best.pt", recursive=True) or glob.glob(f"{MODEL_DIR}/**/*.pt", recursive=True)
if not cands: raise FileNotFoundError(f"Khong thay file .pt trong {MODEL_DIR}")
MODEL_PATH = cands[0]
log(f"Dung model: {MODEL_PATH}")
import torch, cv2, numpy as np
from ultralytics import YOLO
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
log(f"Thiet bi: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "  <-- CHUA BAT GPU!"))
model = YOLO(MODEL_PATH); model.to(DEVICE)
log(f"So lop: {len(model.names)} -> {list(model.names.values())}", "OK")
model.predict(np.zeros((IMGSZ, IMGSZ, 3), np.uint8), imgsz=IMGSZ, verbose=False)
log("Warm-up xong", "OK")
MODEL_LOCK = threading.Lock()

# ---------------- 4. DATABASE ----------------
log("=== BUOC 4/7: Mo database SQLite ===", "STEP")
DB = sqlite3.connect(f"{DATA_DIR}/history.db", check_same_thread=False)
DB.row_factory = sqlite3.Row
DB_LOCK = threading.Lock()
with DB_LOCK:
    DB.execute("""CREATE TABLE IF NOT EXISTS history(
        id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, source TEXT, filename TEXT,
        total INTEGER, counts TEXT, detections TEXT, image TEXT, meta TEXT)""")
    DB.commit()
n_old = DB.execute("SELECT COUNT(*) FROM history").fetchone()[0]
log(f"Database san sang, da co {n_old} ban ghi cu", "OK")

def db_add(source, filename, dets, counts, image_path=None, meta=None):
    with DB_LOCK:
        cur = DB.execute("INSERT INTO history(ts,source,filename,total,counts,detections,image,meta) VALUES(?,?,?,?,?,?,?,?)",
                         (datetime.now().isoformat(timespec="seconds"), source, filename, sum(counts.values()),
                          json.dumps(counts, ensure_ascii=False), json.dumps(dets), image_path, json.dumps(meta or {})))
        DB.commit()
        return cur.lastrowid

def row_to_dict(r, detail=False):
    d = {"id": r["id"], "ts": r["ts"], "source": r["source"], "filename": r["filename"],
         "total": r["total"], "counts": json.loads(r["counts"]), "has_image": bool(r["image"]),
         "meta": json.loads(r["meta"] or "{}")}
    if detail: d["detections"] = json.loads(r["detections"])
    return d

# ---------------- 5. API ----------------
log("=== BUOC 5/7: Khoi tao API ===", "STEP")
from fastapi import FastAPI, UploadFile, WebSocket, WebSocketDisconnect, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse, Response
from starlette.concurrency import run_in_threadpool
import uvicorn, requests

STATS = {"requests": 0, "images": 0, "videos": 0, "ws_clients": 0, "ws_frames": 0, "detections": 0, "errors": 0, "last": "-"}
CLS_COUNT = collections.Counter()
JOBS = {}

app = FastAPI(title="Rice Pest Detection - Full Backend")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
OPEN_PATHS = ("/health", "/docs", "/openapi.json", "/")

@app.middleware("http")
async def guard_and_log(request: Request, call_next):
    t = time.time()
    p = request.url.path
    if API_KEY and request.method != "OPTIONS" and p not in OPEN_PATHS and not p.startswith("/ws"):
        key = request.headers.get("x-api-key") or request.query_params.get("key")
        if key != API_KEY:
            log(f"TU CHOI {request.method} {p} (sai/thieu API key)", "AUTH")
            return JSONResponse({"detail": "Invalid API key"}, status_code=401)
    resp = await call_next(request)
    STATS["requests"] += 1
    STATS["last"] = f"{request.method} {p} {resp.status_code}"
    if p != "/health": log(f"{request.method} {p} -> {resp.status_code} ({(time.time()-t)*1000:.0f}ms)", "HTTP")
    return resp

def count_of(dets):
    return dict(collections.Counter(d["cls"] for d in dets))

def infer(img, conf=CONF, plot=False):
    with MODEL_LOCK:
        r = model.predict(img, conf=conf, imgsz=IMGSZ, verbose=False)[0]
    dets = []
    for b in r.boxes:
        name = model.names[int(b.cls)]
        dets.append({"box": [round(v, 1) for v in b.xyxy[0].tolist()], "cls": name,
                     "cls_id": int(b.cls), "conf": round(float(b.conf), 3)})
        CLS_COUNT[name] += 1
    STATS["detections"] += len(dets)
    h, w = img.shape[:2]
    return {"width": w, "height": h, "detections": dets}, (r.plot() if plot else None)

def decode(data):
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        STATS["errors"] += 1; raise HTTPException(400, "Khong doc duoc anh")
    return img

def check_key_ws(ws):
    return (not API_KEY) or ws.query_params.get("key") == API_KEY

@app.get("/")
def root(): return {"service": "rice-pest-yolo11", "docs": "/docs", "health": "/health"}

@app.get("/health")
def health():
    return {"status": "ok", "device": DEVICE, "classes": model.names, "uptime_s": int(time.time() - T0),
            "auth_required": bool(API_KEY)}

@app.get("/pests")
def pests():
    out = []
    for i, name in model.names.items():
        info = PEST_INFO.get(name, {})
        out.append({"cls_id": i, "cls": name, "name": info.get("name", name),
                    "description": info.get("description", ""), "prevention": info.get("prevention", "")})
    return out

# ----- Anh -----
@app.post("/predict")
async def predict(file: UploadFile, conf: float = CONF, annotated: bool = False, save: bool = True):
    img = decode(await file.read())
    t = time.time()
    res, plot = await run_in_threadpool(infer, img, conf, True)
    res["counts"] = count_of(res["detections"]); res["ms"] = int((time.time() - t) * 1000)
    if save:
        img_path = None
        if res["detections"]:
            tmp = f"{DATA_DIR}/images/tmp_{uuid.uuid4().hex[:6]}.jpg"
            cv2.imwrite(tmp, plot); img_path = tmp
        rid = db_add("image", file.filename, res["detections"], res["counts"], img_path, {"conf": conf, "size": [res["width"], res["height"]]})
        if img_path:
            final = f"{DATA_DIR}/images/{rid}.jpg"; os.replace(img_path, final)
            with DB_LOCK: DB.execute("UPDATE history SET image=? WHERE id=?", (final, rid)); DB.commit()
        res["id"] = rid
    if annotated:
        res["image_b64"] = "data:image/jpeg;base64," + base64.b64encode(cv2.imencode(".jpg", plot)[1]).decode()
    STATS["images"] += 1
    log(f"Anh '{file.filename}' {res['width']}x{res['height']}: {len(res['detections'])} doi tuong {res['counts']} - {res['ms']}ms" + (f" -> luu #{res.get('id')}" if save else ""), "IMAGE")
    return res

# ----- Video -----
def process_video(job_id, path, conf, skip, fname):
    job = JOBS[job_id]
    try:
        cap = cv2.VideoCapture(path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 25
        W, H = int(cap.get(3)), int(cap.get(4))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
        log(f"Job {job_id}: {W}x{H}, {fps:.1f}fps, {total} frame, skip={skip}", "VIDEO")
        raw, final = f"/content/jobs/{job_id}_raw.mp4", f"/content/jobs/{job_id}.mp4"
        writer = cv2.VideoWriter(raw, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
        frames, counts, i, last, next_log = [], collections.Counter(), 0, None, 0.1
        while True:
            ok, frame = cap.read()
            if not ok: break
            if i % skip == 0:
                res, last = infer(frame, conf, True)
                frames.append({"frame": i, "time": round(i / fps, 2), "detections": res["detections"]})
                counts.update(d["cls"] for d in res["detections"])
            writer.write(last if last is not None else frame)
            i += 1; job["progress"] = round(i / total, 3)
            if job["progress"] >= next_log:
                log(f"Job {job_id}: {job['progress']*100:.0f}% ({i}/{total})", "VIDEO"); next_log += 0.1
        cap.release(); writer.release()
        log(f"Job {job_id}: ma hoa H.264...", "VIDEO")
        r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw, "-vcodec", "libx264", "-pix_fmt", "yuv420p", final], capture_output=True, text=True)
        out = final if r.returncode == 0 else raw
        if r.returncode != 0: log(f"ffmpeg loi, dung ban raw: {r.stderr[-200:]}", "WARN")
        counts = dict(counts)
        rid = db_add("video", fname, [], counts, None, {"job_id": job_id, "frames": len(frames), "duration_s": round(total / fps, 1)})
        job.update(status="done", progress=1.0, frames=frames, counts=counts, video=out, history_id=rid)
        STATS["videos"] += 1
        log(f"Job {job_id} XONG -> luu #{rid}. Tong hop: {counts}", "VIDEO")
    except Exception as e:
        STATS["errors"] += 1; job.update(status="error", error=str(e)); log(f"Job {job_id} LOI: {e}", "ERROR")

@app.post("/predict_video")
async def predict_video(file: UploadFile, conf: float = CONF, skip: int = 3):
    job_id = uuid.uuid4().hex[:10]
    path = f"/content/jobs/{job_id}_in.mp4"
    data = await file.read()
    with open(path, "wb") as f: f.write(data)
    log(f"Nhan video '{file.filename}' ({len(data)/1e6:.1f}MB) -> job {job_id}", "VIDEO")
    JOBS[job_id] = {"status": "processing", "progress": 0.0}
    threading.Thread(target=process_video, args=(job_id, path, conf, max(1, skip), file.filename), daemon=True).start()
    return {"job_id": job_id}

@app.get("/job/{job_id}")
def job_status(job_id: str):
    job = JOBS.get(job_id)
    if not job: raise HTTPException(404, "Khong co job")
    return {k: v for k, v in job.items() if k != "video"}

@app.get("/job/{job_id}/video")
def job_video(job_id: str):
    job = JOBS.get(job_id)
    if not job or job.get("status") != "done": raise HTTPException(404, "Chua xong")
    return FileResponse(job["video"], media_type="video/mp4")

# ----- Stream webcam -----
@app.websocket("/ws")
async def ws_stream(ws: WebSocket):
    if not check_key_ws(ws):
        log("TU CHOI ket noi WebSocket (sai API key)", "AUTH"); await ws.close(code=1008); return
    await ws.accept()
    STATS["ws_clients"] += 1
    cid, n, t_start, session = uuid.uuid4().hex[:6], 0, time.time(), collections.Counter()
    log(f"Client stream {cid} ket noi (online: {STATS['ws_clients']})", "WS")
    try:
        while True:
            data = await ws.receive_bytes()
            img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
            if img is None: continue
            t = time.time()
            res, _ = await run_in_threadpool(infer, img)
            res["ms"] = int((time.time() - t) * 1000)
            await ws.send_json(res)
            session.update(d["cls"] for d in res["detections"])
            n += 1; STATS["ws_frames"] += 1
            if n % 30 == 0: log(f"Client {cid}: {n} frame, ~{n/(time.time()-t_start):.1f} fps, infer {res['ms']}ms", "WS")
    except WebSocketDisconnect:
        pass
    except Exception as e:
        STATS["errors"] += 1; log(f"Client {cid} loi: {e}", "ERROR")
    finally:
        STATS["ws_clients"] -= 1
        if n:
            rid = db_add("stream", f"webcam_{cid}", [], dict(session), None, {"frames": n, "duration_s": round(time.time() - t_start, 1)})
            log(f"Client {cid} ngat sau {n} frame -> luu phien #{rid}: {dict(session)}", "WS")

# ----- Lich su -----
@app.get("/history")
def history(limit: int = 20, offset: int = 0, source: str = ""):
    q, args = "SELECT * FROM history", []
    if source: q += " WHERE source=?"; args.append(source)
    q += " ORDER BY id DESC LIMIT ? OFFSET ?"; args += [limit, offset]
    with DB_LOCK:
        rows = DB.execute(q, args).fetchall()
        total = DB.execute("SELECT COUNT(*) FROM history" + (" WHERE source=?" if source else ""), [source] if source else []).fetchone()[0]
    return {"total": total, "items": [row_to_dict(r) for r in rows]}

@app.get("/history/{rid}")
def history_detail(rid: int):
    with DB_LOCK: r = DB.execute("SELECT * FROM history WHERE id=?", (rid,)).fetchone()
    if not r: raise HTTPException(404, "Khong co ban ghi")
    return row_to_dict(r, detail=True)

@app.get("/history/{rid}/image")
def history_image(rid: int):
    with DB_LOCK: r = DB.execute("SELECT image FROM history WHERE id=?", (rid,)).fetchone()
    if not r or not r["image"] or not os.path.exists(r["image"]): raise HTTPException(404, "Khong co anh")
    return FileResponse(r["image"], media_type="image/jpeg")

@app.delete("/history/{rid}")
def history_delete(rid: int):
    with DB_LOCK:
        r = DB.execute("SELECT image FROM history WHERE id=?", (rid,)).fetchone()
        if not r: raise HTTPException(404, "Khong co ban ghi")
        DB.execute("DELETE FROM history WHERE id=?", (rid,)); DB.commit()
    if r["image"] and os.path.exists(r["image"]): os.remove(r["image"])
    log(f"Da xoa ban ghi #{rid}", "DB")
    return {"deleted": rid}

@app.get("/stats")
def stats():
    with DB_LOCK: rows = DB.execute("SELECT ts, source, counts FROM history").fetchall()
    by_cls, by_day, by_src = collections.Counter(), collections.defaultdict(collections.Counter), collections.Counter()
    for r in rows:
        c = json.loads(r["counts"]); by_cls.update(c); by_src[r["source"]] += 1
        by_day[r["ts"][:10]].update(c)
    return {"records": len(rows), "by_source": dict(by_src), "by_class": dict(by_cls.most_common()),
            "by_day": {d: dict(c) for d, c in sorted(by_day.items())}}

@app.get("/export.csv")
def export_csv():
    with DB_LOCK: rows = DB.execute("SELECT * FROM history ORDER BY id").fetchall()
    buf = io.StringIO(); w = csv.writer(buf)
    w.writerow(["id", "time", "source", "filename", "total"] + list(model.names.values()))
    for r in rows:
        c = json.loads(r["counts"])
        w.writerow([r["id"], r["ts"], r["source"], r["filename"], r["total"]] + [c.get(n, 0) for n in model.names.values()])
    return Response("\ufeff" + buf.getvalue(), media_type="text/csv",
                    headers={"Content-Disposition": "attachment; filename=history.csv"})

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
log(f"Dang khoi dong server o cong {PORT}...")
for _ in range(20):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).ok: break
    except Exception: time.sleep(1)
else: raise RuntimeError("Server khong len duoc")
log("Server API da chay", "OK")

# ---------------- 6. TUNNEL ----------------
log("=== BUOC 6/7: Mo tunnel cong khai ===", "STEP")
TUNNEL = {"proc": None, "url": None}

def start_tunnel():
    proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    found = threading.Event(); TUNNEL["url"] = None
    def reader():
        for line in proc.stdout:
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if m and not found.is_set(): TUNNEL["url"] = m.group(0); found.set()
    threading.Thread(target=reader, daemon=True).start()
    TUNNEL["proc"] = proc
    for i in range(30):
        if found.wait(1): break
        if i % 5 == 4: log("Dang cho cloudflared cap link...")
    return TUNNEL["url"]

def check_public(url):
    for i in range(15):
        try:
            if requests.get(url + "/health", timeout=5).ok: return True
        except Exception: pass
        time.sleep(2)
        if i % 3 == 2: log("Link chua san sang, thu lai...")
    return False

def banner(url):
    print("\n" + "=" * 66)
    print(" BACKEND URL :", url)
    print(" WebSocket   :", url.replace("https", "wss") + "/ws" + (f"?key={API_KEY}" if API_KEY else ""))
    print(" Swagger     :", url + "/docs")
    print(" Lich su     :", url + "/history")
    print(" Thong ke    :", url + "/stats")
    print(" Xuat CSV    :", url + "/export.csv")
    print(" API key     :", API_KEY or "(khong khoa)")
    print("=" * 66 + "\n", flush=True)

url = start_tunnel()
if not url: raise RuntimeError("Khong lay duoc link cloudflared")
log(f"Co link: {url}. Kiem tra truy cap ben ngoai...")
log("Link hoat dong" if check_public(url) else "Link chua phan hoi, cho them vai giay", "OK")
banner(url)

# ---------------- 7. GIU SONG ----------------
log("=== BUOC 7/7: Backend dang chay. Giu cell nay, KHONG dong tab. Bam Stop de tat ===", "STEP")
tick = 0
try:
    while True:
        time.sleep(HEARTBEAT); tick += 1
        up = int(time.time() - T0)
        gpu = f" | GPU {torch.cuda.memory_allocated()/1e6:.0f}MB" if DEVICE == "cuda" else ""
        jobs_run = sum(1 for j in JOBS.values() if j["status"] == "processing")
        with DB_LOCK: nrec = DB.execute("SELECT COUNT(*) FROM history").fetchone()[0]
        log(f"Song {up//60}p{up%60:02d}s | req={STATS['requests']} anh={STATS['images']} video={STATS['videos']}(chay {jobs_run}) "
            f"| stream online={STATS['ws_clients']} frame={STATS['ws_frames']} | phat hien={STATS['detections']} | DB={nrec} | loi={STATS['errors']}{gpu}", "HEARTBEAT")
        if tick % 6 == 0:
            if CLS_COUNT: log(f"Thong ke loai (phien nay): {dict(CLS_COUNT.most_common())}", "STATS")
            log(f"Request gan nhat: {STATS['last']}")
        if TUNNEL["proc"].poll() is not None:
            log("Tunnel bi ngat! Mo lai (URL se DOI)...", "WARN")
            url = start_tunnel()
            if url: banner(url)
except KeyboardInterrupt:
    log("Dang tat...", "STOP")
    server.should_exit = True; TUNNEL["proc"].terminate()
    with DB_LOCK: DB.commit(); DB.close()
    log("Da tat. Du lieu da luu tren Drive.", "STOP")
